#Context-Aware Recommender System (CARS) - FUNCTIONS
Collections of functions to work with CARS

Author: Gustavo Fleury Soares

Project: <<https://github.com/gustavofleury/DataSparsity_treat_ContextAware_RS>>

In [ ]:
#Paths
DATASETS_FOLDER = '/content/drive/My Drive/CARS/Datasets/'
COLAB_FOLDER = '/content/drive/My Drive/Colab Notebooks/'

#Grouping method and Metric
LINKAGE_METHOD = 'complete'
LINKAGE_METRIC = 'cosine'

In [ ]:
!apt-get update

In [ ]:
import numpy as np
import pandas as pd
from pandas_profiling import ProfileReport

from __future__ import print_function
from gensim.models import KeyedVectors
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
import re
import os
%matplotlib inline

import math

from scipy.cluster.hierarchy import dendrogram, linkage

import subprocess
import itertools

/usr/local/lib/python3.6/dist-packages/statsmodels/tools/_testing.py:19: FutureWarning: pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.
  import pandas.util.testing as tm


In [ ]:
try:
  import fasttext
  import fasttext.util

except ImportError:
  !git clone https://github.com/facebookresearch/fastText.git
  %cd fastText
  !pip install .
  %cd ~
  import fasttext
  import fasttext.util

In [ ]:
#Install JAVA and CARSKit
!apt-get install -y openjdk-8-jdk-headless -qq > /dev/null      #install openjdk
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"     #set environment variable
!java -version       #check java version
%cd /content/
!git clone https://github.com/irecsys/CARSKit.git
%cd /content/drive/My\ Drive/CARS/Datasets/

In [ ]:
try:
  import pywFM
  %env LIBFM_PATH=/home/libfm/bin/
  %cd /content/drive/My\ Drive/CARS/Datasets/

except ImportError:
  #Lib FM
  !git clone https://github.com/srendle/libfm /home/libfm
  %cd /home/libfm/
  # taking advantage of a bug to allow us to save model #ShameShame
  !git reset --hard 91f8504a15120ef6815d6e10cc7dee42eebaab0f
  !make all

  %env LIBFM_PATH=/home/libfm/bin/
  !pip install git+https://github.com/jfloff/pywFM

  import pywFM
  %cd /content/drive/My\ Drive/CARS/Datasets/

In [ ]:
try:
  import datawig

except ImportError:
  !pip install datawig
  import datawig

In [ ]:
import pickle
def save_obj(obj, name ):
    with open('obj/'+ name + '.pkl', 'wb') as f:
        pickle.dump(obj, f, pickle.HIGHEST_PROTOCOL)

def load_obj(name ):
    with open('obj/' + name + '.pkl', 'rb') as f:
        return pickle.load(f)

In [ ]:
# !pip install -U pandas-profiling

# Word2Vector
Transdorm word(s) in vector

In [ ]:
VECTOR_URL = 'https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/'


In [ ]:
def load_w2v_model(model_name='wiki.en'):
  if os.path.exists(DATASETS_FOLDER + model_name +'.bin'):
    return fasttext.load_model(DATASETS_FOLDER + model_name +'.bin')
  else:
    %cd '{DATASETS_FOLDER}'
    !wget {VECTOR_URL}+{model_name}+.zip
    !unzip {model_name}.zip
    return fasttext.load_model(DATASETS_FOLDER + model_name +'.bin')


In [ ]:
ft = load_w2v_model('dbpedia')

#LOAD WORDS

In [ ]:
def allWordsDF(df, start_column):
  words = []
  for c in df.columns[start_column:]:
    words = np.append(words, c)
    words = np.append(words, df[c][ df[c].notna() ].unique() )
  return(words)

In [ ]:
# def uniqueValuesOfColumn(
#       dfp ):
#     return dfp[dfp.notna()].unique()

def uniqueValuesOfColumn(dfp, add_Column_title_in_each_value=0):
    #append the title of column in each value
    # Example: ['landscape urban', 'landscape mountains', 'landscape country side', 'landscape coast line']
    if add_Column_title_in_each_value > 0:
      result = []
      for element in dfp[dfp.notna()].unique():
        result.append(dfp.name + ' ' + element)
      return np.array(result)

    else:
      # Example: ['urban', 'mountains', 'country side', 'coast line']
      return dfp[dfp.notna()].unique()

In [ ]:
def sent_vectorizer(sent, model):
    sent_vec =[]
    numw = 0
    for w in sent:
        try:
            if numw == 0:
                sent_vec = model.get_word_vector(w)
            else:
                sent_vec = np.add(sent_vec, model.get_word_vector(w))
            numw+=1
        except:
            pass

    return np.asarray(sent_vec) / numw

In [ ]:
def wordListVec(wordList, model):
  wordVec=[]
  for word in wordList:
    #When title has some UpperCase in the middle, add space. Ex: 'DrivingStyle' -> 'Driving Style'
    word = re.sub(r"(\w)([A-Z])", r"\1 \2", word)
    wordVec.append( sent_vectorizer(word.split(), model) )
  return wordVec

In [ ]:
# Creating the tsne to plot
def wv2LowDim(words, model):
  tsne = TSNE(perplexity=30.0, n_components=2, init='pca', n_iter=5000)
  return ( tsne.fit_transform( wordListVec(words, model) ) )

#PLOTTING

In [ ]:
FIGURE_SIZE = 4             #Big:16   #Medium:8   #Small:4

def plot_w2v_dendogram(df_p, model, fig_size=FIGURE_SIZE, consider_title=0):
  qtd_Y = len(df_p)       # math.ceil(len(df.columns[3:])/2)
  fig, axs = plt.subplots(qtd_Y, 2, figsize=(2*fig_size, qtd_Y*fig_size ))

  axis_y=0
  for titleColumn in df_p:
    uniqueWords = uniqueValuesOfColumn(df[titleColumn],add_Column_title_in_each_value=consider_title)
    low_dim_embs = wv2LowDim(uniqueWords, model)

    axs[axis_y,0].set_title(titleColumn)
    for i, label in enumerate(uniqueWords):
        x, y = low_dim_embs[i, :]
        axs[axis_y,0].scatter(x, y)
        axs[axis_y,0].annotate(label,
                  xy=(x, y),
                  xytext=(5, 2),
                  textcoords='offset points',
                  ha='right',
                  va='bottom')

    axs[axis_y,1].set_title("Dendogram " + titleColumn)
    # wordVec = wordListVec(uniqueWords)
    # linkage_dendogram = linkage(wordListVec(uniqueWords, model), method='complete', metric='seuclidean')
    linkage_dendogram = linkage(wordListVec(uniqueWords, model), method=LINKAGE_METHOD, metric=LINKAGE_METRIC)
    dn = dendrogram(
      linkage_dendogram,
      ax=axs[axis_y,1],
      #leaf_rotation=90.,  # rotates the x axis labels
      # leaf_font_size=16.,  # font size for the x axis labels
      orientation='left',
      leaf_label_func=lambda v: str(uniqueWords[v])
    )

    axis_y+=1
  plt.show()

In [ ]:
def plot_w2v_dendogram_Titles(df_p, model, fig_size=FIGURE_SIZE):
  qtd_Y = 1               #len(df_p)       # math.ceil(len(df.columns[3:])/2)
  fig, axs = plt.subplots(qtd_Y, 2, figsize=(2*fig_size, qtd_Y*fig_size ))

  titleColumn= "Columns Titles"
  uniqueWords=[]
  for c in df_p:
    uniqueWords = np.append(uniqueWords, c)
  low_dim_embs = wv2LowDim(uniqueWords, model)

  axs[0].set_title(titleColumn)
  for i, label in enumerate(uniqueWords):
      x, y = low_dim_embs[i, :]
      axs[0].scatter(x, y)
      axs[0].annotate(label,
                xy=(x, y),
                xytext=(5, 2),
                textcoords='offset points',
                ha='right',
                va='bottom')

  axs[1].set_title("Dendogram " + titleColumn)
  # wordVec = wordListVec(uniqueWords)
  linkage_dendogram = linkage(wordListVec(uniqueWords, model), method=LINKAGE_METHOD, metric=LINKAGE_METRIC)
  dn = dendrogram(
    linkage_dendogram,
    ax=axs[1],
    #leaf_rotation=90.,  # rotates the x axis labels
    # leaf_font_size=16.,  # font size for the x axis labels
    orientation='left',
    leaf_label_func=lambda v: str(uniqueWords[v])
  )
  plt.show()

In [ ]:
def plot_w2v(low_dim_embs, labels):
    assert low_dim_embs.shape[0] >= len(labels), "More labels than embeddings"
    fig = plt.figure(figsize=(18, 18))  # in inches
    ax = fig.gca()
    for i, label in enumerate(labels):
        x, y = low_dim_embs[i, :]
        plt.scatter(x, y)
        plt.annotate(label,
                 xy=(x, y),
                 xytext=(5, 2),
                 textcoords='offset points',
                 ha='right',
                 va='bottom')
    ax.set_xticks(np.arange(-500, +500, 50))
    ax.set_yticks(np.arange(-500, +500, 50))
    plt.grid(True)
    plt.show()

#GENERALIZE

In [ ]:
def collect_titleColumn_level(column):
  x = column.split('_Level_')
  if len(x)>1:
    return x[0], str( int(x[1]) + 1 )
  else:
    return x[0], str(1)

In [ ]:
def context_generalization_column(df_p, column, drop_original_column=1, model=ft, use_same_column=0):
  uniqueWords = uniqueValuesOfColumn(df[column])
  if len(uniqueWords) <= 2:  #If there is just 1 value, don't do nothing and return the same DF.
    return df_p

  linkage_dendogram = linkage(wordListVec(uniqueWords, model), method=LINKAGE_METHOD, metric=LINKAGE_METRIC)
  wordPoint_1 = uniqueWords[int(linkage_dendogram[0][0])]
  wordPoint_2 = uniqueWords[int(linkage_dendogram[0][1])]

  titleColumn, level_number = collect_titleColumn_level(column)

  if use_same_column == 1:
    df_p[column] = df_p[column].apply(lambda x: wordPoint_1 + ' ' + wordPoint_2 if (x==wordPoint_1 or x==wordPoint_2) else x)

  else:
    df_p[titleColumn +'_Level_'+ level_number] = df_p[column].apply(lambda x: wordPoint_1 + ' ' + wordPoint_2 if (x==wordPoint_1 or x==wordPoint_2) else x)

    if drop_original_column == 1:
      df_p = df_p.drop(columns=[column])

  return df_p

In [ ]:
def context_generalization_ALLColumns(df_p, model=ft, start_column=3, use_same_column=0):
  for column in df_p.columns[start_column:]:
    df_p = context_generalization_column(df_p, column, drop_original_column=1, use_same_column=use_same_column )
  return df_p

In [ ]:
def context_generalization_ListColumns(df_p, list_column, model=ft):
  for column in list_column:
    df_p = context_generalization_column(df_p, column, use_same_column=1)
  return df_p

In [ ]:
def add_column_title_to_value(df_p, column):
  df_p[column] = df_p[column].fillna('').apply(lambda x: np.nan if x == '' else column + ' ' + x )
  return df_p

In [ ]:
def generalize_group_2_columns(df_p, column1, column2, add_column_title_to_value=0, drop_original_columns=1):
  #Group the 2 Columns ()
  if add_column_title_to_value == 1:
    df_p = add_column_title_to_value(df_p, column1)
    df_p = add_column_title_to_value(df_p, column2)

  df_p[column1 + ' ' + column2] = df_p[[column1, column2]].fillna('').agg(' '.join, axis=1)
  df_p[column1 + ' ' + column2] = df_p[column1 + ' ' + column2].apply(lambda x: np.nan if x == ' ' else x )
  df_p[column1 + ' ' + column2] = df_p[column1 + ' ' + column2].str.strip()
  if drop_original_columns == 1:
    df_p = df_p.drop(columns=[column1,column2])
  return df_p

In [ ]:
def context_generalization_group_columns(df_p, drop_original_columns=1, model=ft):
  uniqueWords=[]
  for c in df_p.iloc[:,3:]:
    uniqueWords = np.append(uniqueWords, c)
  linkage_dendogram = linkage(wordListVec(uniqueWords, model), method=LINKAGE_METHOD, metric=LINKAGE_METRIC)
  column1 = uniqueWords[int(linkage_dendogram[0][0])]
  column2 = uniqueWords[int(linkage_dendogram[0][1])]
  # print(column1, column2 )

  df_p = generalize_group_2_columns(df_p, column1, column2, drop_original_columns=drop_original_columns)
  return df_p

# Data Sparsity Calculation

In [ ]:
def calculate_cardinality(df_p, column_position):
  return len(uniqueValuesOfColumn(df_p.iloc[:,column_position]))

In [ ]:
def calculate_Multiplication_cardinality_Contexts(df_p, column_start):
  # Multiplication of the cardinality of each Context
  # card(C1) x card(C2) x ... x card(Ct)
  multiplication_result = 1
  for column in range(column_start,len(df_p.columns)):
    card_context = calculate_cardinality(df_p, column)
    if card_context > 0:
      multiplication_result = multiplication_result * card_context
  return multiplication_result

In [ ]:
def verify_if_ratings_without_context(df_p,column_start):
  # 1 if there is ratings without any context #Otherwise 0
  # number_columns = len(df.columns)
  # dfx = df_p.copy()
  # for column in range(3,number_columns):
  #   dfx = dfx[ dfx.iloc[:,column].isna() ]

  if len(collect_just_UserItemMatrix(df_p, column_start)) > 0:
    return 1
  else:
    return 0

In [ ]:
def collect_just_UserItemMatrix(df_p, column_start):
  #Take out the Context, and return just the UxI matrix;
  #Don't delete the Ratings with
  dfx = df_p.copy()
  dfx = dfx.drop(columns=df_p.columns[column_start:])
  return dfx.drop_duplicates()

# def collect_just_UserItemMatrix(df_p, column_start):
#   #Take out the Context, and return just the UxI matrix;

#   number_columns = len(df.columns)
#   dfx = df_p.copy()
#   for column in range(3,number_columns):
#     dfx = dfx[ dfx.iloc[:,column].isna() ]
#   return dfx

In [ ]:
def count_ratings(dfp, column_rating):
  return len( dfp[ ~dfp.iloc[:,column_rating].isna() ])

In [ ]:
def delete_columns_without_values(df_p):
  for column in df_p.columns:
    if df_p[column].isna().sum() == len(df_p):
      df_p = df_p.drop(columns=[column])
  return df_p

In [ ]:
def remove_duplicates_rows_NoRating(df_p, rating_column=2):
  # Remove the duplicates Rows not considering the column Rating

  df_noRating = df_p.drop(columns=df.columns[rating_column])

  # Delete the
  repeated_rows_NoRatings = df_noRating.duplicated(keep='first')
  # repeated_rows_NoRatings = df_noRating.duplicated(keep='last')
  # repeated_rows_NoRatings = df_noRating.duplicated(keep=False)

  # print('Number of Duplicate rows (without Rating Column): ', repeated_rows_NoRatings.sum())
  if repeated_rows_NoRatings.sum() == 0:
    return df_p
  else:
    return df_p[~repeated_rows_NoRatings]

In [ ]:
def sparsity(df_p, column_users=0, column_items=1, column_ratings=2, column_start_contexts=3, print_details=0):
  df_p = delete_columns_without_values(df_p)
  # df_p = remove_duplicates_rows_NoRating(df_p, rating_column=column_ratings)
  df_p = df_p.drop_duplicates(keep='last')

  card_users = calculate_cardinality(df_p,column_users)
  card_items = calculate_cardinality(df_p,column_items)
  card_ratings = count_ratings(df_p,column_ratings)

  if len(df_p.columns) > 3:
    multiplication_card_context =calculate_Multiplication_cardinality_Contexts(df_p, column_start_contexts)
    ratings_without_context =  verify_if_ratings_without_context(df_p,column_start_contexts) # 1 if there is ratings without any context #Otherwise 0
    overall_sparsity = 1 - card_ratings/(card_users * card_items * (multiplication_card_context + ratings_without_context) )
  else:
    multiplication_card_context = 'NA'
    ratings_without_context = 'NA'
    overall_sparsity = 1 - card_ratings/(card_users * card_items)

  if print_details == 1:
    print('Cardinalities (Number of distinct values):')
    print('Users:            ', card_users)
    print('Items:            ', card_items)
    print('Ratings:          ', card_ratings)
    print('Multi |Context|s: ', multiplication_card_context)
    print('Overall Sparsity: ', overall_sparsity)
    print('Data Density:     ', 1-overall_sparsity)
    print('---------')
    print('nContexts:', len(df_p.columns)-column_start_contexts)
    print('Has Dimension UxI without Context: ', 'True' if ratings_without_context else 'False')
    print('---------')

  return overall_sparsity

#CARSKit Functions

In [ ]:
def install_java_CARSKit():
  #Install JAVA and CARSKit
  !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null      #install openjdk
  os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"     #set environment variable
  !java -version       #check java version
  %cd /content/
  !git clone https://github.com/irecsys/CARSKit.git
  %cd /content/drive/My\ Drive/CARS/Datasets/

In [ ]:
def CARSKIT(df_p):
  #Call CARSKit Java application, apply the default Configuration
  df_p.to_csv(DATASETS_FOLDER + 'InCarMusic.csv', index=False)
  !java -jar /content/CARSKit/jar/CARSKit-v0.3.5.jar -c /content/drive/My\ Drive/CARS/Datasets/setting-kFold.conf


In [ ]:
def CARSKit_test_kfold(df_p):
  #Call CARSKit Java application, apply the default Configuration
  df_p.to_csv(DATASETS_FOLDER + 'InCarMusic.csv', index=False)
  output = subprocess.check_output ("java -jar /content/CARSKit/jar/CARSKit-v0.3.5.jar -c /content/drive/My\ Drive/CARS/Datasets/setting-kFold.conf | grep Final | awk '{print $11 $13} ' | tr ',' ' ' ", shell=True )
  mae = float(output.split()[0])
  rmse = float(output.split()[1])

  return mae, rmse

In [ ]:
def CARSKit_test(df_p):
  #Call CARSKit Java application, apply the default Configuration
  df_p.to_csv(DATASETS_FOLDER + 'InCarMusic.csv', index=False)
  output = subprocess.check_output ("java -jar /content/CARSKit/jar/CARSKit-v0.3.5.jar -c /content/drive/My\ Drive/CARS/Datasets/setting_Train_Test.conf | grep Final | awk '{print $11 $13} ' | tr ',' ' ' ", shell=True )
  mae = float(output.split()[0])
  rmse = float(output.split()[1])

  return mae, rmse

In [ ]:
def collect_metrics_CARSKit_Generalization(df_p, number_levels_generalization=1):
  #Level 0 (Basic - No Generalization)
  mae, rmse = CARSKit_test(df_p)
  df_CARSKit_Result = pd.DataFrame([[0, ' ', mae, rmse]], columns = ["Level", "Variables", 'MAE', 'RMSE'])

  list_columns = df_p.columns[3:]
  for L in range(1, len(list_columns)+1):
      for subset in itertools.combinations(list_columns, L):
        df_generalized = df_p.copy()
        for generalization_level in range(1,number_levels_generalization+1):
          df_generalized = context_generalization_ListColumns(df_generalized, subset)
          mae, rmse = CARSKit_test(df_generalized)
          print('Level: ', generalization_level, ' Var: ', subset,  ' MAE: ', mae)
          a_series = pd.Series([generalization_level, str(subset), mae, rmse], index = df_CARSKit_Result.columns)
          df_CARSKit_Result =  df_CARSKit_Result.append(a_series, ignore_index=True)

  return df_CARSKit_Result

# SEGMENT U-I

In [ ]:
def reorder_columns(dfp):
  cols = [dfp.columns[-1]] + [col for col in dfp if col != dfp.columns[-1]]
  return dfp[cols]

def merge_2_columns(dfp, column1 = 'UserID', column2 = 'ItemID', delete_Original_Columns = 1):
  df_UI = dfp.copy()
  df_UI['UI'] = df_UI[column1].astype(str) + '-' + df_UI[column2].astype(str)
  if delete_Original_Columns == 1:
    df_UI = df_UI.drop(columns=['UserID', 'ItemID']).drop_duplicates()

  #return reorder_columns(df_UI)
  return df_UI

def plot_heatmap_correlation(dfp):
  plt.figure(figsize=(20,12))
  sns.heatmap(dfp.corr(), annot = True)
  plt.show()

def tranform_matrix_context_values_as_columns_with_ratings(dfp, Ratings_Column = ' Rating', UI_Column = 'UI'):
  RATINGS_COLUMN = Ratings_Column
  df_aux = dfp[[UI_Column]].copy()

  context_columns = [col for col in dfp if col not in [RATINGS_COLUMN, UI_Column]]   #Remove Rating and UI

  # The ratings without CONTEXT will be called 'main'
  df_aux['main'] = np.where( dfp.loc[:,context_columns].isnull().all(axis=1), dfp[RATINGS_COLUMN], np.nan )

  for colName in context_columns:
    for contextValue in dfp[colName].unique():
      if ( str(contextValue) != 'nan' ) :
        df_aux[colName + '-' + contextValue] =  np.where( dfp[colName] == contextValue, dfp[RATINGS_COLUMN], np.nan  )

  df_aux = df_aux.drop_duplicates()

  df_aux2 = df_aux[[UI_Column]].drop_duplicates()
  for columName in df_aux.columns[1:]:
    df_right = df_aux[ df_aux[columName].notna()  ][[UI_Column, columName]].drop_duplicates()
    df_right = df_right.groupby(UI_Column, as_index=False).mean()  # Collect the MEAN value of Ratings
    df_aux2 = df_aux2.merge(df_right, on=UI_Column, how='left')

  return df_aux2

# Sparsity of new UI table:
def basic_sparsity(dfp):
  return (dfp.isna().sum().sum()/ (len(dfp)*len(dfp.columns)))

CORRELATION_TRESHOLD = 0.95
def filter_correlated_columns(dfp, correlation_matrix, column):
  lst_corr_contexts = corr[column][ corr[column] >= CORRELATION_TRESHOLD ].index.tolist()
  lst_corr_contexts.insert(0,'UI')
  if 'main' in lst_corr_contexts: lst_corr_contexts.remove('main')

  df_fc = dfp[lst_corr_contexts] # Filter Correlated columns
  df_fc = df_fc[~df_fc.iloc[:,1:].isnull().all(axis=1)]
  return df_fc

In [ ]:
def lst_lst_of_correlated_contexts(corr, corr_treshold = CORRELATION_TRESHOLD ):
  lst_lst_corr_contexts = []
  for column in corr.columns:
    lst_corr_contexts = corr[column][ ( corr[column] >= corr_treshold ) ].index.tolist()
    if len(lst_corr_contexts) > 1:
      lst_lst_corr_contexts.append( lst_corr_contexts )
  return lst_lst_corr_contexts

CORRELATION_TRESHOLD = 0.99
def filter_columns(dfp1, lst_columns):
  dfp = dfp1.copy()
  lst_columns.insert(0,'UI')  if 'UI' not in lst_columns else lst_columns
  dfp = dfp[lst_columns].copy() # Filter Correlated columns
  dfp = dfp[~dfp.iloc[:,1:].isnull().all(axis=1)]
  return dfp

# FILL NA with moda in CORRELATED Contexts Values:
def input_value_to_nan(dfp):
  dfp_T = dfp.transpose()
  for colName in dfp_T.columns[:]:
    inputValue = dfp_T[colName].iloc[1:].mode()[0]  #mode
    dfp_T[colName].fillna(inputValue, inplace=True)
  return dfp_T.transpose()

#Transform
def transform_back_Context_to_Compact(dfp, dfp_Ori):
  dfp_aux = df.iloc[0:0,:].copy()

  #Incialize Context with NaN values
  dict_contexts = {}

  for i, row in dfp.iterrows():
    (UserID, ItemID) = row['UI'].split('-')

    for colName in row.index[1:]:
      #Set Contexts to NAN
      for colName1 in df.columns[3:]:
        dict_contexts[colName1] = np.nan

      (contextTitle, contextValue) = colName.split('-')
      dict_contexts[contextTitle] = contextValue
      Rating = np.nan if math.isnan(row[colName]) else int(row[colName])

      list_Values = [int(UserID), int(ItemID), Rating, dict_contexts['DrivingStyle'], dict_contexts['landscape'],
                  dict_contexts['mood'], dict_contexts['naturalphenomena '], dict_contexts['RoadType'],
                  dict_contexts['sleepiness'], dict_contexts['trafficConditions'], dict_contexts['weather'] ]
      dfp_aux.loc[len(dfp_aux)] = list_Values

  return dfp_aux

In [ ]:
def CARSKit_test_REPEAT(dfp, num_repetitions=1):
  MAE_lst, RMSE_lst = [], []
  for _ in range(0,num_repetitions):
    ( MAE_seg, RMSE_seg ) = CARSKit_test(dfp)
    MAE_lst.append(MAE_seg)
    RMSE_lst.append(RMSE_seg)
  return (MAE_lst, RMSE_lst)

def collect_ALL_MAE_SPARSITY(dfp1, correlation_treshold_value = CORRELATION_TRESHOLD, imputation_option='MOST_FREQUENT', num_repetitions=1 ):
  dfp=dfp1.copy()
  df_UI = merge_2_columns(dfp, 'UserID', 'ItemID', delete_Original_Columns = 1)
  df_UI_context = tranform_matrix_context_values_as_columns_with_ratings(df_UI, Ratings_Column = ' Rating')

  corr = df_UI_context.corr()
  lst_lst_corr = lst_lst_of_correlated_contexts(corr, correlation_treshold_value)

  df_ALL_inputed = dfp.copy()
  df_ALL_better = dfp.copy()
  SPARSITY_SEGMENT_BASIC = 0
  SPARSITY_IMPUTATED_OVERALL = sparsity(dfp)
  ( MAE_Ori, RMSE_Ori ) = CARSKit_test_REPEAT(dfp, num_repetitions)

  dict_CARSKIT_Results = {}
  dict_CARSKIT_Results['Original'] = ('Original', SPARSITY_SEGMENT_BASIC, SPARSITY_IMPUTATED_OVERALL, MAE_Ori, RMSE_Ori )
  print(dict_CARSKIT_Results)
  print('Length List of Correlated Columns: ', len(lst_lst_corr))

  for lst_col in lst_lst_corr:

    # Filter Correlated Columns
    df_fc = filter_columns(df_UI_context, lst_col)

    # Input Values in Correlated
    if imputation_option == 'MOST_FREQUENT':
      df_fc_inputed = input_value_to_nan(df_fc)
    elif imputation_option =='DATAWIG':
      df_fc_inputed = datawig.SimpleImputer.complete(df_fc)
    elif imputation_option =='FM':
      df_fc_inputed = input_value_using_FM(df_fc)

    # Transform Back to Original Format
    df_fc_inputed = df_fc_inputed.drop_duplicates()
    df_fc_inputed_Compact = transform_back_Context_to_Compact(df_fc_inputed, df)
    df_fc_inputed_Compact = df_fc_inputed_Compact.drop_duplicates()

    # Test MAE/RMSE:
    print(lst_col)
    ( MAE_seg, RMSE_seg ) = CARSKit_test_REPEAT(dfp.append(df_fc_inputed_Compact), num_repetitions)
    SPARSITY_SEGMENT_BASIC = basic_sparsity(df_fc)
    SPARSITY_IMPUTATED_OVERALL = sparsity(dfp.append(df_fc_inputed_Compact))
    print( SPARSITY_IMPUTATED_OVERALL, ' - ', MAE_seg, ' - ',  RMSE_seg)
    dict_CARSKIT_Results[ ', '.join(lst_col) ] = ('Segment', SPARSITY_SEGMENT_BASIC, SPARSITY_IMPUTATED_OVERALL, MAE_seg, RMSE_seg )

    # Append to Completed Inputed
    df_ALL_inputed = df_ALL_inputed.append(df_fc_inputed_Compact)
    print('All_inputed:')
    SPARSITY_SEGMENT_BASIC = basic_sparsity(pd.concat([df_ALL_inputed, df_train, df_train]).drop_duplicates(keep=False))
    SPARSITY_IMPUTATED_OVERALL = sparsity(df_ALL_inputed)
    ( MAE, RMSE ) = CARSKit_test_REPEAT(df_ALL_inputed, num_repetitions )
    print(SPARSITY_IMPUTATED_OVERALL, '- ',  MAE, ' - ',  RMSE)
    dict_CARSKIT_Results['ALL-' + ', '.join(lst_col) ] = ('ALL', SPARSITY_SEGMENT_BASIC, SPARSITY_IMPUTATED_OVERALL, MAE, RMSE )

    # # Append to Just Better Results
    # if MAE_seg <= MAE_Ori:
    #   df_ALL_better = df_ALL_better.append(df_fc_inputed_Compact)
    #   print('All_Better:')
    #   SPARSITY_SEGMENT_BASIC = basic_sparsity(pd.concat([df_ALL_better, df_train, df_train]).drop_duplicates(keep=False))
    #   SPARSITY_IMPUTATED_OVERALL = sparsity(df_ALL_better)
    #   ( MAE, RMSE ) = CARSKit_test(df_ALL_better)
    #   print(SPARSITY_IMPUTATED_OVERALL, '- ',  MAE, ' - ',  RMSE)
    #   dict_CARSKIT_Results['BETTER-' + ', '.join(lst_col) ] = ('BETTER', SPARSITY_SEGMENT_BASIC, SPARSITY_IMPUTATED_OVERALL, MAE, RMSE )

  return dict_CARSKIT_Results, lst_lst_corr

In [ ]:
def create_dict_for_initial_of_values(dfp):
  # Collect First Letter of Column Title as UPPER  # First letter of value of column as LOWER
  dict_columns_title_values_initials = {}

  for column_name in dfp.columns[3:]:
    #Column Title
    first_letters = [ w[0] for w in column_name.split() ]
    dict_columns_title_values_initials[column_name] = ''.join(first_letters).upper()

    #Column Values
    for uniqueValuesOfColumn in dfp[column_name][dfp[column_name].notna() ].unique():
      first_letters = [ w[0] for w in uniqueValuesOfColumn.split() ]
      dict_columns_title_values_initials[uniqueValuesOfColumn] = ''.join(first_letters).lower()

  # print(dict_columns_title_values_initials)
  return dict_columns_title_values_initials

def replace_names_for_initial(dict_titles_initial, lst_lst_corr):
  #Create Initials list for use in PLOTs;
  lst_lst_corr_INITIALS = []
  for lst_corr_col in lst_lst_corr[:]:
    lst_corr_col_I =[]
    for corr_col in lst_corr_col:
      for title, initial in dict_titles_initial.items():
        corr_col = corr_col.replace(title, initial)

      lst_corr_col_I.append(corr_col)
    if 'UI' in lst_corr_col_I: lst_corr_col_I.remove('UI')
    lst_lst_corr_INITIALS.append(','.join(lst_corr_col_I) )

  # print(len(lst_lst_corr[:]))
  # print((lst_lst_corr_INITIALS))

  return lst_lst_corr_INITIALS

In [ ]:
def subplot_base(y, x, x_ori, title='', y_ticks='' ):
  plt.plot(y, x, label='Imputed Values')
  print(x)
  if x_ori != 0:
    plt.plot(y, [x_ori for i in range(len(x))], '--', color='darkorange', label='Original' )

  plt.plot(y, [min(x) for i in range(len(x))], 'g--' , label='Max/Min')
  plt.plot(y, [max(x) for i in range(len(x))], 'g--' )
  plt.title(title, fontsize=20)
  plt.xlabel('Correlated Columns', fontsize=12)
  plt.legend(loc='upper right', fontsize=12)
  if len(y_ticks) == len(y):
    plt.xticks(y, y_ticks, rotation=90)

def collect_list_results_from_dict(dict_results):
  #Transform the MAE Results DICT in LISTs (for PLOT ou PANDAS)
  x_sparsity_segment = []
  x_sparsity = []
  x_mae = []
  x_rmse = []

  x_sparsity_segment_A = []
  x_sparsity_A = []
  x_mae_A = []
  x_rmse_A = []


  for (t,sparsity_seg, sparsity_imputated, MAE, RMSE) in dict_results.values():
    if ( t == 'Original'):
      x_sparsity_segment_Original = sparsity_seg
      x_sparsity_Original = sparsity_imputated
      x_mae_Original = MAE
      x_rmse_Original = RMSE

    elif ( t == 'Segment'):
      x_sparsity_segment.append(sparsity_seg)
      x_sparsity.append(sparsity_imputated)
      x_mae.append(MAE)
      x_rmse.append(RMSE)

    elif ( t == 'ALL' ) :
      x_sparsity_segment_A.append(sparsity_seg)
      x_sparsity_A.append(sparsity_imputated)
      x_mae_A.append(MAE)
      x_rmse_A.append(RMSE)

  return x_sparsity_segment, x_sparsity, x_mae, x_rmse, x_sparsity_segment_A, x_sparsity_A, x_mae_A, x_rmse_A, x_sparsity_segment_Original, x_sparsity_Original, x_mae_Original, x_rmse_Original

def tranform_dict_results_in_DF(dict_results):
  # Make Lists
  x_sparsity_segment, x_sparsity, x_mae, x_rmse, x_sparsity_segment_A, x_sparsity_A, x_mae_A, x_rmse_A, x_sparsity_segment_Original, x_sparsity_Original, x_mae_Original, x_rmse_Original = collect_list_results_from_dict(dict_results)
  dfp = pd.DataFrame()
  dfp['x_sparsity_segment'] = x_sparsity_segment
  dfp['x_sparsity'] = x_sparsity
  dfp['x_mae'] = x_mae
  dfp['x_rmse'] = x_rmse
  dfp['x_sparsity_segment_A'] = x_sparsity_segment_A
  dfp['x_sparsity_A'] = x_sparsity_A
  dfp['x_mae_A'] = x_mae_A
  dfp['x_rmse_A'] = x_rmse_A
  return dfp

def plot_ALL_MAE_SPARSITY(dict_results, x_ticks_Initials = ''):
  #Collect Dict info like list
  x_sparsity_segment, x_sparsity, x_mae, x_rmse, x_sparsity_segment_A, x_sparsity_A, x_mae_A, x_rmse_A, x_sparsity_segment_Original, x_sparsity_Original, x_mae_Original, x_rmse_Original = collect_list_results_from_dict(dict_results)
  #PLOT Results:

  Y = range(len(x_mae))

  fig = plt.figure(figsize=(25,30))

  plt.subplot(3,2,1)
  subplot_base(Y, x_mae, x_mae_Original, 'MAE', x_ticks_Initials)

  plt.subplot(3,2,2)
  subplot_base(Y, x_rmse, x_rmse_Original, 'RMSE', x_ticks_Initials)

  plt.subplot(3,2,3)
  subplot_base(Y, x_mae_A, x_mae_Original, 'MAE - Cumulative')

  plt.subplot(3,2,4)
  subplot_base(Y, x_rmse_A, x_rmse_Original, 'RMSE - Cumulative')

  plt.subplot(3,2,5)
  subplot_base(Y, x_sparsity_segment, x_sparsity_segment_Original, 'Sparsity Segment', x_ticks_Initials)

  plt.subplot(3,2,6)
  subplot_base(Y, x_sparsity, x_sparsity_Original, 'Overall Sparsity', x_ticks_Initials)

  plt.tight_layout()
  plt.show()

# LIbFM


In [ ]:
def input_value_using_FM(df_fc1):
  df_fc1_INPUTED = df_fc1.copy()

  lst_col1 = df_fc1.columns.to_list()
  lst_col1.remove('UI') if 'UI' in lst_col1 else lst_col1
  for colum_treat in lst_col1:
    df_fc1_INPUTED = df_fc1_INPUTED.append ( input_value_from_MODEL(df_fc1, colum_treat) )

  return df_fc1_INPUTED

fm = pywFM.FM(task='regression', num_iter=10, seed=1)

def input_value_from_MODEL(dfp, col_treat):
  df_fc_train = dfp[ dfp[col_treat].notna() ]
  features_cols = df_fc_train.columns.to_list()
  features_cols.remove('UI')
  features_cols.remove(col_treat)

  X = df_fc_train.loc[:,features_cols] # Leave just VARIABLES in X.

  #Remove ROW with all NAN (in Features)
  df_fc_train = df_fc_train[ ~X.isna().all(axis=1) ]

  X = df_fc_train.loc[:,features_cols].fillna(0).to_numpy()
  y = df_fc_train[col_treat].to_numpy()

  #TEST
  df_fc_test = dfp[ ~dfp[col_treat].notna() ]
  X_test = df_fc_test.loc[:,features_cols]

  #Remove ROW with all NAN (in Features)
  df_fc_test = df_fc_test[ ~X_test.isna().all(axis=1) ]
  X_test = df_fc_test.loc[:,features_cols].fillna(0).to_numpy()
  y_test = np.full((len(X_test)), 0) # Just fake '0's values - Don't influence in TRAIN MODEL calculation

  #Apply the Model
  model = fm.run(X, y, X_test, y_test)

  #Input values to DF
  df_fc_test_inputted = df_fc_test.copy()
  df_fc_test_inputted[col_treat] = np.asarray(model.predictions )

  return df_fc_test_inputted